# Chord-Length Extraction from Binary Masks  *(template)*

This notebook reads **binary segmentation masks** (black background, white
droplets) from a microscopy image and measures the **chord lengths** of the
white objects along the X and Y directions. The chord lengths are a proxy for
droplet size. Results are saved to a `.npy` file for the companion processing /
plotting notebooks.

**Input you need:** a folder of segmentation masks

---

### How to use this notebook (for non-coders)
1. Run the cells **from top to bottom** (Shift + Enter).
2. You only need to edit **one cell**: the one marked **"EDIT THIS CELL"**.
   Lines to change are marked `# <-- CHANGE THIS`.
3. The processing cell shows a preview plot for every image so you can check the
   masks and chords look sensible.


## Step 0 — Load the Python tools

Imports the libraries used below: OpenCV (`cv2`) to read images, NumPy for
arrays, Matplotlib to plot, **PoreSpy** to compute chord lengths, and
scikit-image for object labelling. Just run it.

*First time only:* if you get a `ModuleNotFoundError`, install with
`pip install opencv-python numpy matplotlib porespy scikit-image scipy`.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import porespy as ps
import scipy.ndimage as spim
from skimage.measure import regionprops

## Step 1 — Helper function: count individual chord lengths

PoreSpy gives a *binned* distribution, but we also want the raw length of every
individual chord. This small function labels each separate chord in the
chord-image and returns its length (its filled area, in pixels). You don't need
to change anything here.

In [ ]:
# Returns the length (in pixels) of every individual chord in a chord image.
def chord_counts(im):
    labels, N = spim.label(im > 0)
    props = regionprops(labels)
    chord_lens = np.array([i.filled_area for i in props])
    return chord_lens

## Step 2 — Configuration  ✏️ **EDIT THIS CELL**

The **only** cell you need to change. Set the folder containing your masks, the
name of the output file, and the list of sample-name tokens used to group your
images.

In [ ]:
# === EDIT THIS CELL ====================================================

# Full/absolute path to the folder that CONTAINS your binary masks.
experiment_dir = "PASTE_YOUR_MASKS_FOLDER_HERE"        # <-- CHANGE THIS
# Example (original): ".../PhenDC3/Nikon_Replicate1/02_Masks/"

# Name of the .npy file to create (saved one level ABOVE experiment_dir).
# Tip: encode the condition + replicate number, e.g. 'Radius_PhenDC3_r3.npy'
output_name = "Radius_condition_r1.npy"                # <-- CHANGE THIS

# Sample-name tokens. Each mask filename must contain ONE of these strings;
# the script uses them to group images by sample. Edit to match YOUR filenames.
# (e.g. add '12nt' if you have that sample: ['T1', 'T2', 'T4', '12nt'])
sample_tokens = ["T1", "T2", "T4"]                     # <-- CHANGE THIS
# ========================================================================

## Step 3 — Find and group the mask files by sample

Moves into your masks folder and sorts every filename into the right sample
group based on the `sample_tokens` above. The printout lets you confirm that the
right files landed in the right group **before** running the (slower) analysis.

In [ ]:
os.chdir(experiment_dir)
all_files = sorted(os.listdir())

# Group files by sample token: keep only files whose name contains the token.
files_by_sample = {token: sorted(f for f in all_files if token in f)
                   for token in sample_tokens}

for token, flist in files_by_sample.items():
    print(f'{token}: {len(flist)} file(s)')
    for f in flist:
        print(f'   {f}')

## Step 4 — Extract chord lengths from every mask

The main loop. For each sample and each image it:
1. reads the mask and turns it into a strict black/white (0/255) image,
2. lays down **chords** along X and along Y (PoreSpy `apply_chords`),
3. computes both the **binned distribution** (`chord_length_distribution`) and
   the **individual chord lengths** (`chord_counts`), and
4. draws a 4-panel preview: the mask, the X-chords, the Y-chords, and the
   chord-length histogram.

Everything is stored in the nested dictionary `cld[sample][replicate]`. This is
the slow step — one set of panels appears per image.

In [ ]:
cld = {}
for sample_label, sample_files in files_by_sample.items():
    cld[sample_label] = {}
    print(sample_label)
    for filename, rep in zip(sample_files, range(1, len(sample_files) + 1)):
        print(filename, rep)
        cld[sample_label][rep] = {'x': [], 'y': [], 'count_x': [], 'count_y': []}

        # Read file as single image, force to a clean 0/255 binary mask.
        img_raw = cv2.imread(filename)
        img = (img_raw[:, :, 0] > 0).astype(np.uint8) * 255

        # Lay down chords along each axis.
        crds_x = ps.filters.apply_chords(im=img, spacing=1, axis=0, trim_edges=False)
        crds_y = ps.filters.apply_chords(im=img, spacing=1, axis=1, trim_edges=False)

        # Binned chord-length distribution.
        data_x = ps.metrics.chord_length_distribution(crds_x, bins=100)
        data_y = ps.metrics.chord_length_distribution(crds_y, bins=100)

        # Store binned distributions and the raw per-chord lengths.
        cld[sample_label][rep]['x'] = data_x
        cld[sample_label][rep]['y'] = data_y
        cld[sample_label][rep]['count_x'] = chord_counts(crds_x)
        cld[sample_label][rep]['count_y'] = chord_counts(crds_y)

        # Preview plot: mask, X chords, Y chords, and the distribution.
        plt.subplots(1, 4, figsize=(20, 5))
        plt.subplot(141)
        plt.imshow(img)
        plt.title(f'{sample_label} - rep {rep}')
        plt.subplot(142)
        plt.imshow(crds_x, interpolation='none', origin='lower')
        plt.title('Chords - X')
        plt.axis(False)
        plt.subplot(143)
        plt.imshow(crds_y, interpolation='none', origin='lower')
        plt.title('Chords - Y')
        plt.axis(False)
        plt.subplot(144)
        plt.bar(x=data_y.L, height=data_y.cdf, width=data_y.bin_widths, color='b', edgecolor='k', alpha=0.5, label='Y')
        plt.bar(x=data_x.L, height=data_x.cdf, width=data_x.bin_widths, color='r', edgecolor='k', alpha=0.5, label='X')
        plt.xlabel("Chord Length Distribution")
        plt.ylabel("Frequency")
        plt.legend(frameon=False)
        plt.show()

## Step 5 — Quick check: which samples were processed?

Lists the top-level keys of the results dictionary, so you can confirm every
expected sample is present before saving.

In [ ]:
cld.keys()

## Step 6 — Save the results

Writes the `cld` dictionary to your `output_name` file. Note: because Step 3
changed the working directory into `experiment_dir`, the `"../"` means the file
is saved **one folder above** your masks folder. Move/rename it afterwards if you
prefer a different location.

In [ ]:
# Save results in NPY format for the processing & plotting notebooks.
np.save("../" + output_name, cld, allow_pickle=True)

## Step 7 — Re-load to confirm the file is valid

Reads the file you just saved back into memory and peeks at the keys of one
entry. If this runs without error, the save worked. (Edit the sample label below
if your first sample is not `T1`.)

In [ ]:
# Load saved results to check they were correctly saved.
cld_bulk_single = np.load("../" + output_name, allow_pickle=True).item()
first_sample = sample_tokens[0]          # <-- or type a label, e.g. 'T1'
cld_bulk_single[first_sample][1].keys()